# Random Forest Training (V4)
This notebook trains:
- one binary quality model (is_good: 1 good / 0 bad)
- one model per active error reason (err_partial_rom, err_too_fast, err_torso_sway, err_asymmetry)
- validation metrics for each trainable active error model when a stratified split is possible
- feature-importance summaries for the quality model and each trained active error model


In [1]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


In [2]:
DATASET_FILE = "reps_dataset_v2.csv"
MODEL_FILE = "bicep_curl_model.pkl"
TEST_SIZE = 0.2
RANDOM_STATE = 42
MIN_ROWS_FOR_SPLIT = 10
FEATURE_IMPORTANCE_TOP_N = 5

FEATURE_COLUMNS = [
    "min_left_angle",
    "max_left_angle",
    "min_right_angle",
    "max_right_angle",
    "rep_duration",
    "left_rom",
    "right_rom",
    "elbow_rom_diff",
    "concentric_duration",
    "eccentric_duration",
    "left_peak_velocity",
    "right_peak_velocity",
    "torso_lean_mean",
    "torso_lean_max",
    "torso_sway",
    "left_elbow_drift",
    "right_elbow_drift",
    "pose_visibility_mean",
    "pose_visibility_min",
    "tracking_lost_ratio",
]

ERROR_COLUMNS = [
    "err_partial_rom",
    "err_too_fast",
    "err_torso_sway",
    "err_asymmetry",
]

TARGET_COLUMN = "is_good"


In [3]:
def build_random_forest():
    return RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )


In [4]:
def can_do_stratified_split(y):
    return y.nunique() >= 2 and len(y) >= MIN_ROWS_FOR_SPLIT and y.value_counts().min() >= 2


In [5]:
def get_feature_importance_records(model, feature_columns):
    importance_frame = pd.DataFrame(
        {
            "feature": feature_columns,
            "importance": model.feature_importances_,
        }
    ).sort_values("importance", ascending=False, ignore_index=True)
    return importance_frame.to_dict(orient="records")


In [6]:
def print_feature_importances(model_name, feature_importances, top_n=FEATURE_IMPORTANCE_TOP_N):
    print(f"\n{model_name} top feature importances:")
    for item in feature_importances[:top_n]:
        print(f"  {item['feature']}: {item['importance']:.4f}")


In [7]:
def evaluate_binary_model(model_name, model, X_train, y_train, X_test=None, y_test=None):
    metrics = {
        "train_accuracy": round(float(model.score(X_train, y_train)), 4),
        "test_accuracy": None,
        "classification_report": None,
        "confusion_matrix": None,
        "evaluated_on_test_split": X_test is not None and y_test is not None,
    }

    print(f"{model_name} train accuracy:", metrics["train_accuracy"])

    if not metrics["evaluated_on_test_split"]:
        print(f"{model_name} test metrics skipped (insufficient balanced data).")
        return metrics

    y_pred = model.predict(X_test)
    metrics["test_accuracy"] = round(float(accuracy_score(y_test, y_pred)), 4)
    metrics["classification_report"] = classification_report(
        y_test,
        y_pred,
        zero_division=0,
    )
    metrics["confusion_matrix"] = confusion_matrix(y_test, y_pred).tolist()

    print(f"{model_name} test accuracy:", metrics["test_accuracy"])
    print(f"\n{model_name} classification report:")
    print(metrics["classification_report"])
    print(f"{model_name} confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    return metrics


In [8]:
df = pd.read_csv(DATASET_FILE)
print("Dataset loaded:", DATASET_FILE)
print("Rows:", len(df))

required_columns = FEATURE_COLUMNS + ERROR_COLUMNS + [TARGET_COLUMN]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN]).copy()
df[ERROR_COLUMNS] = df[ERROR_COLUMNS].fillna(0).astype(int)
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)

print("\nFeature scaling not applied: Random Forest models are tree-based and do not require normalized inputs.")

print("\nQuality label counts (is_good):")
print(df[TARGET_COLUMN].value_counts().sort_index())

print("\nActive reason label counts:")
for col in ERROR_COLUMNS:
    counts = df[col].value_counts().sort_index().to_dict()
    print(f"{col}: {counts}")

df[FEATURE_COLUMNS + ERROR_COLUMNS + [TARGET_COLUMN]].head()


Dataset loaded: reps_dataset_v2.csv
Rows: 49

Feature scaling not applied: Random Forest models are tree-based and do not require normalized inputs.

Quality label counts (is_good):
is_good
0    27
1    22
Name: count, dtype: int64

Active reason label counts:
err_partial_rom: {0: 42, 1: 7}
err_too_fast: {0: 39, 1: 10}
err_torso_sway: {0: 45, 1: 4}
err_asymmetry: {0: 43, 1: 6}


,min_left_angle,max_left_angle,min_right_angle,max_right_angle,rep_duration,left_rom,right_rom,elbow_rom_diff,concentric_duration,eccentric_duration,...,left_elbow_drift,right_elbow_drift,pose_visibility_mean,pose_visibility_min,tracking_lost_ratio,err_partial_rom,err_too_fast,err_torso_sway,err_asymmetry,is_good
0,13.87,177.79,17.97,179.98,7.61,163.91,162.02,1.89,6.92,0.62,...,0.0222,0.0235,0.9890,0.9797,0.0,0,0,0,0,1
1,18.98,177.70,25.16,179.98,3.43,158.73,154.82,3.90,3.24,0.14,...,0.0065,0.0125,0.9821,0.9651,0.0,0,1,0,0,0
2,8.48,179.96,16.86,179.50,4.69,171.48,162.64,8.85,4.07,0.58,...,0.0101,0.0257,0.9795,0.9571,0.0,0,0,0,1,0
3,12.91,179.76,18.00,179.99,8.96,166.85,161.99,4.86,8.43,0.48,...,0.0772,0.0946,0.9904,0.9455,0.0,0,0,0,0,1
4,0.38,179.98,13.19,179.86,3.84,179.60,166.67,12.93,3.71,0.08,...,0.0536,0.0356,0.9662,0.8791,0.0,0,1,0,0,0


In [9]:
X = df[FEATURE_COLUMNS]
y_quality = df[TARGET_COLUMN]

quality_metrics = None
quality_feature_importances = []
can_train_quality = y_quality.nunique() >= 2

if can_do_stratified_split(y_quality):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_quality,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_quality,
    )
    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)
else:
    X_train, y_train = X, y_quality
    X_test, y_test = None, None
    print("Not enough balanced data for a stratified test split.")
    print("Training quality model on full dataset and skipping test metrics.")


Train shape: (39, 20)
Test shape: (10, 20)


In [10]:
quality_model = None

if can_train_quality:
    quality_model = build_random_forest()
    quality_model.fit(X_train, y_train)
    print("Quality model trained.")
else:
    print("Quality model not trained: only one class found in is_good.")


Quality model trained.


In [11]:
if quality_model is not None:
    quality_metrics = evaluate_binary_model(
        "Quality",
        quality_model,
        X_train,
        y_train,
        X_test,
        y_test,
    )
    quality_feature_importances = get_feature_importance_records(
        quality_model,
        FEATURE_COLUMNS,
    )
    print_feature_importances("Quality", quality_feature_importances)


Quality train accuracy: 1.0
Quality test accuracy: 0.7

Quality classification report:
              precision    recall  f1-score   support

           0       0.80      0.67      0.73         6
           1       0.60      0.75      0.67         4

    accuracy                           0.70        10
   macro avg       0.70      0.71      0.70        10
weighted avg       0.72      0.70      0.70        10

Quality confusion matrix:
[[4 2]
 [1 3]]

Quality top feature importances:
  eccentric_duration: 0.1397
  left_peak_velocity: 0.0903
  pose_visibility_mean: 0.0649
  left_rom: 0.0644
  right_peak_velocity: 0.0639


In [12]:
error_models = {}
error_metrics = {}
error_feature_importances = {}

for error_col in ERROR_COLUMNS:
    y_error = df[error_col]

    if y_error.nunique() < 2:
        constant_value = int(y_error.iloc[0])
        error_models[error_col] = {
            "type": "constant",
            "value": constant_value,
        }
        error_metrics[error_col] = {
            "type": "constant",
            "value": constant_value,
            "train_accuracy": 1.0,
            "test_accuracy": None,
            "classification_report": None,
            "confusion_matrix": None,
            "evaluated_on_test_split": False,
        }
        error_feature_importances[error_col] = []
        print(f"{error_col}: only one class ({constant_value}), saved as constant predictor.")
        continue

    if can_do_stratified_split(y_error):
        X_error_train, X_error_test, y_error_train, y_error_test = train_test_split(
            X,
            y_error,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y_error,
        )
        print(f"\n{error_col}: train/test split enabled.")
    else:
        X_error_train, y_error_train = X, y_error
        X_error_test, y_error_test = None, None
        print(f"\n{error_col}: insufficient balanced data for a stratified test split.")

    model = build_random_forest()
    model.fit(X_error_train, y_error_train)

    feature_importances = get_feature_importance_records(
        model,
        FEATURE_COLUMNS,
    )
    metrics = evaluate_binary_model(
        error_col,
        model,
        X_error_train,
        y_error_train,
        X_error_test,
        y_error_test,
    )

    error_models[error_col] = {
        "type": "model",
        "model": model,
    }
    error_metrics[error_col] = metrics
    error_feature_importances[error_col] = feature_importances

    print(f"{error_col}: model trained.")
    print_feature_importances(error_col, feature_importances)



err_partial_rom: train/test split enabled.
err_partial_rom train accuracy: 1.0
err_partial_rom test accuracy: 1.0

err_partial_rom classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00         1

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

err_partial_rom confusion matrix:
[[9 0]
 [0 1]]
err_partial_rom: model trained.

err_partial_rom top feature importances:
  min_right_angle: 0.2329
  right_rom: 0.2043
  min_left_angle: 0.0912
  left_rom: 0.0860
  elbow_rom_diff: 0.0500

err_too_fast: train/test split enabled.
err_too_fast train accuracy: 1.0
err_too_fast test accuracy: 0.9

err_too_fast classification report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94         8
           1       1.00     

In [13]:
bundle = {
    "version": "v4_binary_quality_plus_active_reason_validation",
    "dataset_file": DATASET_FILE,
    "feature_columns": FEATURE_COLUMNS,
    "error_columns": ERROR_COLUMNS,
    "target_column": TARGET_COLUMN,
    "quality_model": quality_model,
    "quality_metrics": quality_metrics,
    "quality_feature_importances": quality_feature_importances,
    "error_models": error_models,
    "error_metrics": error_metrics,
    "error_feature_importances": error_feature_importances,
}

joblib.dump(bundle, MODEL_FILE)
print(f"\nSaved model bundle to: {MODEL_FILE}")



Saved model bundle to: bicep_curl_model.pkl


## Notes
- Only the active reason labels err_partial_rom, err_too_fast, err_torso_sway, and err_asymmetry are trained and evaluated in this version.
- Feature scaling is intentionally skipped because these Random Forest models are tree-based and do not require normalized inputs.
- Built-in Random Forest feature importances are saved in the model bundle for both the quality model and the active error models.
- SHAP is not included yet; the built-in feature importances give a dependency-free first pass, and SHAP can be added later for deeper per-sample explanations.
